# **400m Career Data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff
%matplotlib inline
from datetime import datetime
import re
from nssstats.plots import std_plot
from nssstats.plots import iqr_plot
from nssstats.plots import quadrant_plot, half_plot
from ipywidgets import interact, FloatSlider
from nssstats.cm import cm_analysis
from sklearn.model_selection import train_test_split
from scipy.stats import probplot
from scipy.stats import t, sem
from scipy.stats import chi2
from statsmodels.stats.proportion import proportion_confint
import statsmodels.formula.api as sm

In [ ]:
sprinters = pd.read_csv("../data/Worlds_Fastest_Sprinters_Stats.csv")

# **Data** **Basics**

In [ ]:
sprinters.head()

In [ ]:
sprinters.info

In [ ]:
sprinters.shape

In [ ]:
print(sprinters.dtypes)


In [ ]:
sprinters.describe()


In [ ]:
sprinters.corr


In [ ]:
sprinters.isnull().sum()


# **General** **EDA**

Let's make a column for the total PR time

In [ ]:
sprinters['Total_Time_PRs'] = round(sprinters['100_PR'] + sprinters['200_PR'] + sprinters['400_PR'],2)
sprinters.head()

Let's make a column for the total career average time

In [ ]:
sprinters['Total_Time_SB_Avg'] = round(sprinters['Avg_Season_Best_100m'] + sprinters['Avg_Season_Best_200m'] + sprinters['Avg_Season_Best_400m'],2)
sprinters.head()

Let's make a column for the actual age of the athletes

In [ ]:
# Ensure the DOB column is in datetime format
sprinters['DOB'] = pd.to_datetime(sprinters['DOB'], errors='coerce')  # Coerce will handle invalid dates as NaT

# Get the current year
current_year = datetime.now().year

# Calculate the age by subtracting the birth year from the current year
sprinters['Age'] = current_year - sprinters['DOB'].dt.year

# Display the updated DataFrame with the new 'Age' column
print(sprinters[['DOB', 'Age']].head())


Let's add the sprinter's photo to the database by merging it with the photo csv

In [ ]:
sprinter_photo = pd.read_csv("../data/Sprinter_Photo.csv")

In [ ]:
sprinter_photo.head(3)

In [ ]:
sprinters = pd.merge(sprinters, sprinter_photo, on=['Athlete'],how='left')
sprinters.head(3)

Let's add second database to join number of season to each events dataframe (For Years Competed in each event).

In [ ]:
sprinters_df2 = pd.read_csv("../data/Worlds_Fastest_Sprinters_Master_List_Yearly_Progression.csv")
sprinters_df2.head(3)

In [ ]:
yrs_competed_400m = sprinters_df2[sprinters_df2['Event'] == '400m'].groupby('Athlete').size().reset_index(name='Years')

In [ ]:
yrs_competed_400m = yrs_competed_400m.sort_values(by='Years', ascending=False)
yrs_competed_400m.head()

In [ ]:
yrs_competed_400m = pd.DataFrame(yrs_competed_400m)
yrs_competed_400m.head(3)

Use this line of code to find a specific athlete in the data frame

In [ ]:
#yrs_competed_400m.loc[(yrs_competed_400m.Athlete == 'Frankie Fredericks')]

In [ ]:
#yrs_competed_400m.isnull()

In [ ]:
#yrs_competed_400m_nulls = yrs_competed_400m[yrs_competed_400m.isnull().any(axis=1)]
#print(yrs_competed_400m_nulls)

In [ ]:
yrs_competed_400m

In [ ]:
#Extract Excel file for data fix if needed

'''
xlwriter = pd.ExcelWriter('400M_Years_Competed.xlsx')
yrs_competed_400m.to_excel(xlwriter, sheet_name='Years Competed 400m')
xlwriter.close()
'''

Let's add a third database which incorportae's every race in each athletes's career.

In [ ]:
sprinters_df3 = pd.read_csv("../data/Sprinter_Career.csv")
sprinters_df3.head(3)

In [ ]:
All_400m_Races = sprinters_df3[sprinters_df3['Event'] == '400m']
All_400m_Races.head(3)

Let's drop all the races that were DNS, DNF, or DQ

In [ ]:
All_400m_Races = All_400m_Races[~All_400m_Races['Time'].isin(['DNS', 'DNF', 'DQ'])]
All_400m_Races.head(3)

Let's make sure that the time column is now a numeric datatype

In [ ]:
All_400m_Races['Time'] = pd.to_numeric(All_400m_Races['Time'], errors='coerce')

Let's drop all the Indoor marks

In [ ]:
All_400m_Races = All_400m_Races[All_400m_Races['Meet_Type'] != 'Indoor']
All_400m_Races.head(3)

Let's drop times that aren't legal from the dataframe

In [ ]:
All_400m_Races = All_400m_Races[All_400m_Races['Legal'] != 'NO']
All_400m_Races.head(3)

Let's Change any non-numeric values to either blank or 0 in the Wind column

To changes the non-numeric values values to blank

In [ ]:
#All_400m_Races['Wind'] = All_400m_Races['Wind'].replace('NWI',np.nan)

To chabge the non-numeric valuesvalues to 0

In [ ]:
#All_400m_Races['Wind'] = All_400m_Races['Wind'].replace('NWI',0)

If I need to change the blank / null values in the wind column to 0

In [ ]:
#All_400m_Races['Wind'] = All_400m_Races['Wind'].fillna(0)

Let's make sure that the wind column is now a numeric datatype

In [ ]:
All_400m_Races['Wind'] = pd.to_numeric(All_00m_Races['Wind'], errors='coerce')

Let's drop all races that have Illegal wind (> 2.0)

In [ ]:
#All_400m_Races = All_400m_Races[All_400m_Races['Wind'] > 2.0]
#All_400m_Races.head(3)

Let's get the Caeer Average for Each Athlete

In [ ]:
Career_average_400m = All_400m_Races.groupby('Athlete')['Time'].mean().reset_index(name='Career_Avg_400m')

In [ ]:
Career_average_400m = pd.DataFrame(Career_average_400m)
Career_average_400m.head(3)

In [ ]:
#All_400m_Races = pd.merge(All_400m_Races, Career_average_400m, on=['Athlete'],how='left')
#All_400m_Races.head(3)

Seasons Best For Each Athlete (Alternative Option)

In [ ]:
Season_Bests_400m = All_400m_Races.groupby(["Athlete", "Year"])["Time"].min()

Avg_Season_Best_400m = (Season_Bests_400m.groupby("Athlete").mean().reset_index().rename(columns={"Time": "Avg_Season_Best_400m"}))

Avg_Season_Best_400m

In [ ]:
Avg_Season_Best_400m = pd.DataFrame(Avg_Season_Best_400m)
Avg_Season_Best_400m.head(3)

Total Races for Each Athlete

In [ ]:
athlete_race_count_400m = All_400m_Races.groupby('Athlete').size().reset_index(name='total_races_400m')
athlete_race_count_400m.head(3)

In [ ]:
athlete_race_count_400m = pd.DataFrame(athlete_race_count_400m)
athlete_race_count_400m.head(3)

In [ ]:
athlete_race_count_400m

Number of races for each athlete by year

In [ ]:
#athlete_race_count_per_year_400m = hundred_meter_races.groupby(['Athlete', 'Year']).size().reset_index(name='races_per_year_100m')
#athlete_race_count_per_year_400m.head(3)

Average Wind For Each Athlete

Note: We may not need the wind factor for the 400m analysis.

In [ ]:
Average_Wind_400m = All_100m_Races.groupby('Athlete')['Wind'].mean().reset_index(name='Avg_Wind_400m')
Average_Wind_100m.head(3)

In [ ]:
Average_Wind_400m = pd.DataFrame(Average_Wind_400m)
Average_Wind_400m.head(3)

Total Wins Per Athlete

In [ ]:
Wins_400m = All_400m_Races[All_400m_Races['Place'] == 1].groupby('Athlete').size().reset_index(name='Wins_400m')
Wins_400m

In [ ]:
Wins_400m = pd.DataFrame(Wins_400m)
Wins_400m.head(3)

Average Place (Finish) For Each Athlete

In [ ]:
Average_Place_400m = All_400m_Races.groupby('Athlete')['Place'].mean().reset_index(name='Avg_Place_400m')
Average_Place_400m.head(3)

Let's Find the Average Place when the Race is a Final

In [ ]:
Final_Heat_400m = All_400m_Races[All_400m_Races['Race'] =='F']
Final_Heat_400m.head(3)

In [ ]:
Average_Place_400mF = Final_Heat_400m.groupby('Athlete')['Place'].mean().reset_index(name='Avg_Place_400m_Final_Heat')
Average_Place_400mF.head(3)

In [ ]:
Average_Place_400mF = pd.DataFrame(Average_Place_400mF)
Average_Place_400mF.head(3)

Let's Merge this with the Average 400m Place Dataframe

In [ ]:
Average_Place_400m = pd.merge(Average_Place_400m, Average_Place_400mF, on=['Athlete'],how='left')
Average_Place_400m.head(3)

Let's Find the Average Place when the Race is a Final in a Championship or High Level Meet

In [ ]:
High_Level_Meet = ['OW','DF', 'DL', 'GW', 'GL']

In [ ]:
High_Level_Meet_Final = All_400m_Races[(All_400m_Races['Race'] =='F') & (All_400m_Races['Category'].isin(High_Level_Meet))]
High_Level_Meet_Final.head(3)

In [ ]:
High_Level_Meet_Final = pd.DataFrame(High_Level_Meet_Final)
High_Level_Meet_Final.head(3)

In [ ]:
Avg_Clutch_Finish_400mF = High_Level_Meet_Final.groupby('Athlete')['Place'].mean().reset_index(name='Avg_Clutch_Finish_400mF')
Avg_Clutch_Finish_400mF.head(3)

Let's Merge this with the Average 400m Place Dataframe as well

In [ ]:
Average_Place_400m = pd.merge(Average_Place_400m, Avg_Clutch_Finish_400mF, on=['Athlete'],how='left')
Average_Place_400m.head(3)

In [ ]:
Average_Place_400m = pd.DataFrame(Average_Place_400m)
Average_Place_400m.head(3)

Total World Athletics Score

In [ ]:
Total_WA_Score_400m = All_400m_Races.groupby('Athlete')['Score'].sum().reset_index(name='Total_WA_Score_400m')
Total_WA_Score_400m.head(3)

In [ ]:
Total_WA_Score_100m = pd.DataFrame(Total_WA_Score_100m)
Total_WA_Score_100m.head(3)

Average World Athletics Score

In [ ]:
Avg_WA_Score_400m = All_400m_Races.groupby('Athlete')['Score'].mean().reset_index(name='Avg_WA_Score_400m')
Avg_WA_Score_400m.head(3)

In [ ]:
Avg_WA_Score_400m = pd.DataFrame(Avg_WA_Score_400m)
Avg_WA_Score_400m.head(3)

## World Athletics Category Rankings (Move New Data Up Here)

| Place | OW  | DF  | GW  | GL  | A   | B   | C   | D   | E   | F   |
|-------|-----|-----|-----|-----|-----|-----|-----|-----|-----|-----|
| 1st   | 350 | 240 | 200 | 170 | 140 | 100 | 60  | 40  | 25  | 15  |
| 2nd   | 310 | 210 | 170 | 145 | 120 | 80  | 50  | 35  | 21  | 10  |
| 3rd   | 280 | 185 | 150 | 130 | 110 | 70  | 45  | 30  | 18  | 5   |
| 4th   | 250 | 170 | 140 | 120 | 100 | 60  | 40  | 25  | 15  |     |
| 5th   | 230 | 155 | 130 | 110 | 90  | 55  | 35  | 22  | 12  |     |
| 6th   | 215 | 145 | 120 | 100 | 80  | 50  | 30  | 19  | 10  |     |
| 7th   | 200 | 135 | 110 | 90  | 70  | 45  | 27  | 17  |     |     |
| 8th   | 185 | 125 | 100 | 80  | 60  | 40  | 25  | 15  |     |     |
| 9th   | 130 | 90  | 70  | 60  |     |     |     |     |     |     |
| 10th  | 120 | 80  | 60  | 50  |     |     |     |     |     |     |
| 11th  | 110 | 70  | 50  | 45  |     |     |     |     |     |     |
| 12th  | 100 | 60  | 45  | 40  |     |     |     |     |     |     |
| 13th  | 95  |     |     |     |     |     |     |     |     |     |
| 14th  | 90  |     |     |     |     |     |     |     |     |     |
| 15th  | 85  |     |     |     |     |     |     |     |     |     |
| 16th  | 80  |     |     |     |     |     |     |     |     |     |

*Source: [World Athletics – Basics of the World Rankings](https://worldathletics.org/world-ranking-rules/basics)*

## Category Scale: Log-Transformed, Rescaled to 1–10

Using the raw 1st-place Placing Scores directly isn't ideal for averaging — they span **15 to 350, roughly a 23x range**, which is too wide and skewed to treat as a linear scale. A pure ordinal scale (OW=10, DF=9, GW=8...) fixes the range problem but assumes every category is equally far apart, which isn't true either — the gap between OW and DF is much bigger than the gap between D and E.

**The approach:** take the log of each category's 1st-place point value, then linearly rescale that log range onto a clean 1–10 scale (OW=10, F=1). This compresses the extreme top-end gap (OW/DF) while still preserving that higher tiers are worth meaningfully more, and it keeps mid/low tiers (B, C, D) from collapsing too close to 1.

| Category | 1st-place points | Scaled score (1–10) |
|----------|------------------|----------------------|
| OW       | 350              | 10.0                 |
| DF       | 240              | 8.9                  |
| GW       | 200              | 8.4                  |
| GL       | 170              | 7.9                  |
| A        | 140              | 7.4                  |
| B        | 100              | 6.4                  |
| C        | 60               | 5.0                  |
| D        | 40               | 3.8                  |
| E        | 25               | 2.5                  |
| F        | 15               | 1.0                  |

**Trade-off to keep in mind:** this scale spreads scores across the *full* 1–10 range fairly evenly, so a runner who mostly races Category B/C still lands mid-scale rather than near the bottom. If you instead want the scale to sharply reward *only* top-tier competition (and treat everything below GL as roughly similar), a linear rescale of the raw points — without the log step — would compress B through F much closer to 1.

In [ ]:
#Scaled Score Mapping dictionary
log_scaled_score_map = {
    'OW': 10.0,
    'DF': 8.9,
    'GW': 8.4,
    'GL': 7.9,
    'A': 7.4,
    'B': 6.4,
    'C': 5.0,
    'D': 3.8,
    'E': 2.5,
    'F': 1.0
}

# Create new column
All_400m_Races['Categorey_Log_Scaled_Score'] = All_400m_Races['Category'].map(log_scaled_score_map)

In [ ]:
Avg_Race_Cateogry_Score_Log = All_400m_Races.groupby('Athlete')['Categorey_Log_Scaled_Score'].mean().reset_index(name='Avg_Log_Cat_Score_400m')
Avg_Race_Cateogry_Score_Log.head(3)

In [ ]:
Avg_Race_Cateogry_Score_Log = pd.DataFrame(Avg_Race_Cateogry_Score_Log)
Avg_Race_Cateogry_Score_Log.head(3)

## Category Scale: Pure Linear Rescale to 1–10

This version rescales the raw 1st-place Placing Scores directly onto a 1–10 range (OW=10, F=1), without the log-compression step. Because it preserves the full ~23x ratio between OW (350) and F (15), it produces a much more top-heavy scale: GL through F cluster in the bottom third, while OW and DF remain far apart at the top.

| Category | 1st-place points | Scaled score (1–10, linear) |
|----------|------------------|------------------------------|
| OW       | 350              | 10.0                         |
| DF       | 240              | 7.0                          |
| GW       | 200              | 6.0                          |
| GL       | 170              | 5.2                          |
| A        | 140              | 4.4                          |
| B        | 100              | 3.3                          |
| C        | 60               | 2.2                          |
| D        | 40               | 1.7                          |
| E        | 25               | 1.3                          |
| F        | 15               | 1.0                          |

**Use this version if:** you want the ranking system to sharply reward top-tier competition (OW/DF/GW) and treat mid-to-lower tiers as roughly comparable to each other.

In [ ]:
#Scaled Score Mapping dictionary
linear_scaled_score_map = {
    'OW': 10.0,
    'DF': 7.0,
    'GW': 6.0,
    'GL': 5.2,
    'A': 1.4,
    'B': 3.3,
    'C': 2.2,
    'D': 1.7,
    'E': 1.3,
    'F': 1.0
}

# Create new column
All_400m_Races['Categorey_Linear_Scaled_Score'] = All_400m_Races['Category'].map(linear_scaled_score_map)

In [ ]:
Avg_Race_Cateogry_Score_Lin = All_400m_Races.groupby('Athlete')['Categorey_Lin_Scaled_Score'].mean().reset_index(name='Avg_Lin_Cat_Score_400m')
Avg_Race_Cateogry_Score_Lin.head(3)

In [ ]:
Avg_Race_Cateogry_Score_Lin = pd.DataFrame(Avg_Race_Cateogry_Score_Lin)
Avg_Race_Cateogry_Score_Lin.head(3)

In [ ]:
athlete_category_summary = (
    All_400m_Races.groupby(['Athlete', 'category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

athlete_category_summary['win_pct'] = (
    athlete_category_summary['wins']
    / athlete_category_summary['total_races']
    * 100
).round(1)

athlete_category_summary.drop(columns='wins', inplace=True)

In [ ]:
athlete_category_summary_400m = pd.DataFrame(athlete_category_summary)
athlete_category_summary_400m.head(3)

If we want to pivot this table to 1 row per athlete

In [ ]:
pivot = athlete_category_summary.pivot(
    index='Athlete',
    columns='Category',
    values='win_pct'
)

Year By Year Comparision

In [ ]:
athlete_category_year_summary = (
    All_400m_Races.groupby(['Athlete', 'Year', 'category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

athlete_category_year_summary['win_pct'] = (
    athlete_category_year_summary['wins']
    / athlete_category_year_summary['total_races']
    * 100
).round(1)

athlete_category_year_summary.drop(columns='wins', inplace=True)

In [ ]:
athlete_category_year_summary_400m = pd.DataFrame(athlete_category_year_summary)
athlete_category_year_summary_400m.head(3)

Pivot Table Version of This

In [ ]:
pivot = athlete_category_year_summary.pivot_table(
index=['Athlete', 'category'],
columns='Year',
values='win_pct'
)

Athlete's overall stats each year regardless of category

In [ ]:
athlete_year_summary = (
    All_400m_Races.groupby(['Athlete', 'Year'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

athlete_year_summary['win_pct'] = (
    athlete_year_summary['wins']
    / athlete_year_summary['total_races']
    * 100
).round(1)


In [ ]:
athlete_year_summary_400m = pd.DataFrame(athlete_year_summary)
athlete_year_summary_400m.head(3)

Let's Analyze Each Athletes Race Stats By Country

In [ ]:
country_category_summary = (
    All_400m_Races.groupby(['Athlete', 'Country', 'category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

country_category_summary['win_pct'] = (
    country_category_summary['wins']
    / country_category_summary['total_races']
    * 100
).round(1)

country_category_summary.drop(columns='wins', inplace=True)

In [ ]:
country_category_summary_400m = pd.DataFrame(country_category_summary)
country_category_summary_400m.head(3)

Year By Year Analysis (Country)

In [ ]:
country_year_summary = (
    All_400m_Races.groupby(['Athlete', 'Country', 'Year'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

country_year_summary['win_pct'] = (
    country_year_summary['wins']
    / country_year_summary['total_races']
    * 100
).round(1)

country_year_summary.drop(columns='wins', inplace=True)

In [ ]:
country_year_summary_400m = pd.DataFrame(country_year_summary)
country_year_summary_400m.head(3)

Let's Analyze Each Athletes Race Stats By Continent

In [ ]:
continent_category_summary = (
    All_400m_Races.groupby(['Athlete', 'Continent', 'category'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

continent_category_summary['win_pct'] = (
    continent_category_summary['wins']
    / continent_category_summary['total_races']
    * 100
).round(1)

continent_category_summary.drop(columns='wins', inplace=True)

In [ ]:
continent_category_summary_400m = pd.DataFrame(continent_category_summary)
continent_category_summary_400m.head(3)

Year By Year Analysis (Continent)

In [ ]:
continent_year_summary = (
    All_400m_Races.groupby(['Athlete', 'Continent', 'Year'])
      .agg(
          total_races=('Event', 'count'),
          avg_wind=('Wind', 'mean'),
          avg_place=('Place', 'mean'),
          avg_score=('Score', 'mean'),
          wins=('Place', lambda x: (x == 1).sum())
      )
      .reset_index()
)

continent_year_summary['win_pct'] = (
    continent_year_summary['wins']
    / continent_year_summary['total_races']
    * 100
).round(1)

continent_year_summary.drop(columns='wins', inplace=True)

In [ ]:
continent_year_summary_400m = pd.DataFrame(continent_year_summary)
continent_year_summary_400m.head(3)

Compact Version (Country & Continent)

In [ ]:
athlete_country_year_category = (
    All_400m_Races.groupby(
        ['Athlete', 'Country', 'Continent', 'Year', 'category']
    )
    .agg(
        total_races=('Event', 'count'),
        avg_wind=('Wind', 'mean'),
        avg_place=('Place', 'mean'),
        avg_score=('Score', 'mean'),
        wins=('Place', lambda x: (x == 1).sum())
    )
    .reset_index()
)

athlete_country_year_category['win_pct'] = (
    athlete_country_year_category['wins']
    / athlete_country_year_category['total_races']
    * 100
).round(1)

athlete_country_year_category.drop(columns='wins', inplace=True)

In [ ]:
athlete_country_year_category_400m = pd.DataFrame(athlete_country_year_category)
athlete_country_year_category_400m.head(3)

Meerge Data Season and total races data next

In [ ]:
seasons_and_races_400m = pd.merge(athlete_race_count_400m, yrs_competed_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

Let's add the career average for each athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Career_average_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

In [ ]:
seasons_and_races_400m['Avg_Races_Year_400m'] = round(seasons_and_races_400m['total_races_400m'] / seasons_and_races_400m['Years'],2)
sprinters.head()

Let's add the season best average (alternative version) for each athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Avg_Season_Best_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

Let's add the average wind for each athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Average_Wind_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

Let's add total wins for each athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Wins_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

Let's add average place (finish) for each athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Average_Place_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

Let's add the total World Athletics Score for Each Athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Total_WA_Score_400m, on=['Athlete'],how='left')
seasons_and_races_400m.head(3)

Let's add the average World Athletics Score for Each Athlete

In [ ]:
seasons_and_races_400m = pd.merge(seasons_and_races_400m, Avg_WA_Score_400m, on=['Athlete'],how='left')
seasons_and_races_100m.head(3)

In [ ]:
seasons_and_races_400m = pd.DataFrame(seasons_and_races_400m)
seasons_and_races_400m.head(3)

In [ ]:
seasons_and_races_400m

Fixing Null Data

In [ ]:
seasons_and_races_400m.at[9, "Years"] = 1
seasons_and_races_400m.at[9, "Avg_Races_Year_400m"] = 1
seasons_and_races_400m.at[29, "Years"] = 3
seasons_and_races_400m.at[29, "Avg_Races_Year_400m"] = 1
seasons_and_races_400m.at[32, "Years"] = 10
seasons_and_races_400m.at[32, "Avg_Races_Year_400m"] = 5.40
seasons_and_races_400m.at[44, "Years"] = 11
seasons_and_races_400m.at[44, "Avg_Races_Year_400m"] = 8.27
seasons_and_races_400m.at[48, "Years"] = 10
seasons_and_races_400m.at[48, "Avg_Races_Year_400m"] = 1.20

In [ ]:
seasons_and_races_400m

In [ ]:
# We don't need the 'Avg_Season_Best_400m' from the df_400m now that we have it coming from the 'seasons_and_races' df
df_400m= sprinters[['Athlete', 'Country','Continent','Status', 'DOB','Year Born','Month Born','Decade Born','400_PR','T25_400_All_Time_Rank','T25_400_AT_RK_NUM']]
df_400m

In [ ]:
df_400m['SB_Avg_400m_PR_Diff'] = round(df_400m['Avg_Season_Best_400m'] - df_400m['400_PR'],2)
df_400m.head(3)

In [ ]:
df_400m = pd.merge(df_400m, seasons_and_races_400m, on=['Athlete'],how='left')
df_400m.head(3)

In [ ]:
df_400m = df_400m.rename(columns={'Years': 'Years_Competed_400m'})
df_400m.head(3)

In [ ]:
df_400m['Career_Avg_400m_PR_Diff'] = round(df_400m['Career_Avg_400m'] - df_400m['400_PR'],2)
df_400m.head(3)

In [ ]:
df_400m['Win_Pct_400m'] = round((df_400m['Wins_400m'] / df_400m['total_races_400m'])*100,2)
df_400m.head(3)

In [ ]:
#df_400m = df_400m.sort_values(by='Avg Season Best 400m', ascending=False)
#df_400m

In [ ]:
#df_400m = df_400m.sort_values(by='Avg_Season_Best_400m')
#df_400m

In [ ]:
df_400m = df_400m.dropna(subset='400_PR')
df_400m

Fixing Null Values

In [ ]:
df_400m.at[12, "total_races_400m"] = 3
df_400m.at[12, "Years_Competed_400m"] = 3
df_400m.at[12, "Career_Avg_400m"] = 47.793333
df_400m.at[12, "Avg_Races_Year_400m"] = 1.00
df_400m.at[12, "Career_Avg_400m_PR_Diff"] = .08
df_400m.at[28, "total_races_400m"] = 1
df_400m.at[28, "Years_Competed_400m"] = 1
df_400m.at[28, "Career_Avg_400m"] = 46.280000
df_400m.at[28, "Avg_Races_Year_400m"] = 1.00
df_400m.at[28, "Career_Avg_400m_PR_Diff"] = 0
df_400m.at[39, "total_races_400m"] = 3
df_400m.at[39, "Years_Competed_400m"] = 2
df_400m.at[39, "Career_Avg_400m"] = 46.903333
df_400m.at[39, "Avg_Races_Year_400m"] = 1.50
df_400m.at[39, "Career_Avg_400m_PR_Diff"] = 1.03
df_400m.at[44, "total_races_400m"] = 91
df_400m.at[44, "Years_Competed_400m"] = 11
df_400m.at[44, "Career_Avg_400m"] = 45.598901
df_400m.at[44, "Avg_Races_Year_400m"] = 8.27
df_400m.at[44, "Career_Avg_400m_PR_Diff"] = 2.57

In [ ]:
df_400m

In [ ]:
df_400m = df_400m.sort_values(by='Career_Avg_400m')
df_400m

In [ ]:
df_400m = pd.DataFrame(df_400m)

# ***Statistical Analysis 400m***

In [ ]:
from scipy.stats import zscore
from sklearn.linear_model import LinearRegression

Linear Regression

In [ ]:
# Function to calculate standard deviation (consistency)
df_400m['consistency'] = df_400m['Avg_Season_Best_400m']  # Placeholder: You could replace with actual std per year data if available

### 1. Regression Analysis ###
# Linear regression: relationship between average_time and years_competed
X = df_400m[['Years_Competed_400m', '400_PR']]
y = df_400m['Avg_Season_Best_400m']

# Fit model
model = LinearRegression()
model.fit(X, y)

# Predictions and residuals
df_400m['predicted_time'] = model.predict(X)
df_400m['residuals'] = df_400m['Avg_Season_Best_400m'] - df_400m['predicted_time']

print("Regression coefficients (slope):", model.coef_)
print("Intercept:", model.intercept_)


Z-Score Standardization

Version 1

In [ ]:
'''
# Z-score for average_time and years_competed
df_400m['z_time'] = zscore(df_400m['Avg Season Best 400m'])
df_400m['z_years'] = zscore(df_400m['Years_Competed_400m'])
df_400m['z_PR'] = zscore(df_400m['400_PR'])
df_400m['z_career_avg_season'] = zscore(df_400m['Career_Avg_400m'])
df_400m['z_total_races'] = zscore(df['total_races_400m'])
df_400m['z_races_per_year'] = zscore(df['Avg_Races_Year_400m'])

# Z-score comparison (combine time and years)
df_400m['z_combined'] = (df_400m['z_time'] + df_400m['z_years'] + df_400m['z_PR']+ df_400m['z_career_avg_season'] + df_400m['z_total_races'] + df_400m['z_races_per_year']) / 6

'''

Version 2 (Takes into the account the that the total races and avg races per year has a major affect on the combined z-score)

In [ ]:
# Z-score for average_time and years_competed
df_400m['z_time'] = zscore(df_400m['Avg_Season_Best_400m'],nan_policy='omit')
df_400m['z_years'] = -zscore(df_400m['Years_Competed_400m'],nan_policy='omit')
df_400m['z_PR'] = zscore(df_400m['400_PR'],nan_policy='omit')
df_400m['z_career_avg_season'] = zscore(df_400m['Career_Avg_400m'],nan_policy='omit')
df_400m['z_total_races'] = -zscore(df_400m['total_races_400m'],nan_policy='omit')
df_400m['z_races_per_year'] = -zscore(df_400m['Avg_Races_Year_400m'],nan_policy='omit')
#Need to determine if the following below need a positive or negative z-socre
#df_400m['z_avg_wind_per_year'] = -zscore(df_400m['Average_Wind_400m'],nan_policy='omit')
#df_400m['z_avg_place_per_race'] = -zscore(df_400m['Average_Place_400m'],nan_policy='omit')
#df_400m['z_avg_place_per_final'] = -zscore(df_400m['Avg_Place_400m_Final_Heat'],nan_policy='omit')
#df_400m['z_avg_clutch_finish'] = -zscore(df_400m['Avg_Clutch_Finish_400mF'],nan_policy='omit')
#df_400m['z_total_WA_score'] = -zscore(df_400m['Total_WA_Score_400m'],nan_policy='omit')
#df_400m['z_avg_WA_score'] = -zscore(df_400m['Avg_WA_Score_400m'],nan_policy='omit')

# Z-score comparison (combine time and years)
df_400m['z_combined'] = (df_400m['z_time'] + df_400m['z_years'] + df_400m['z_PR']+ df_400m['z_career_avg_season'] + df_400m['z_total_races'] + df_400m['z_races_per_year']) / 6

Alternate Ranking

In [ ]:
df_400m['alt_rank'] = df_400m['z_combined'].rank(ascending=True)

In [ ]:
df_400m.sort_values('alt_rank').head(10)

In [ ]:
Z_Score_Ranking_400m = df_400m.sort_values('alt_rank')

In [ ]:
Z_Score_Ranking_400m = pd.DataFrame(Z_Score_Ranking_400m)
Z_Score_Ranking_400m.head(3)

In [ ]:
#Let's make the entire dataframe a figure factory table


fig =  ff.create_table(Z_Score_Ranking_400m)
fig.show()

#fig.write_html("Z_Score_Ranking_400m.html")
#fig.write_image("Z_Score_Ranking_400m.svg")

Efficiency / Ratio Analysis

In [ ]:
# Efficiency score (average_time per year competed)
df_400m['efficiency_score'] = df_400m['Avg_Season_Best_400m'] / df_400m['Years_Competed_400m']

# Efficiency score: How close the sprinter's average season best to their personal best
df_400m['efficiency_score_sb'] = df_400m['400_PR'] / df_400m['Avg_Season_Best_400m']

# Efficiency score: How close the sprinter's average is to their personal best
df_400m['efficiency_score_pr'] = df_400m['400_PR'] / df_400m['Career_Avg_400m']

Ranking System

In [ ]:
# Combine rankings based on average_time, consistency, and longevity (years_competed)
df_400m['rank_personal_best'] = df_400m['400_PR'].rank(ascending=True)  # Lower personal best is better
df_400m['rank_average_sb'] = df_400m['Avg_Season_Best_400m'].rank(ascending=True)  # Lower is better
df_400m['rank_career_avg'] = df_400m['Career_Avg_400m'].rank(ascending=True)
#df_400m['rank_consistency'] = df_400m['consistency'].rank(ascending=True)  # Lower std dev is better
df_400m['rank_consistency'] = df_400m['consistency'].abs().rank(ascending=True) # Lower residuals (consistency) is better
df_400m['rank_years_competed'] = df_400m['Years_Competed_400m'].rank(ascending=False)  # Longer careers are better
df_400m['rank_total_races'] = df_400m['total_races_400m'].rank(ascending=False)  # More races is better
df_400m['rank_races_per_year'] = df_400m['Avg_Races_Year_400m'].rank(ascending=False)  # More races per year is better
#df_400m['rank_avg_wind_per_year'] = df_400m['Average_Wind_400m'].rank(ascending=False)  # More wind is better
df_400m['rank_avg_place_per_race'] = df_400m['Average_Place_400m'].rank(ascending=True)  # Lower finish each race is better
#df_400m['rank_avg_place_per_final'] = df_400m['Avg_Place_400m_Final_Heat'].rank(ascending=True)  # Lower finish each race is better
#df_400m['rank_avg_clutch_finish'] = df_400m['Avg_Clutch_Finish_400mF'].rank(ascending=True)  # Lower finish each race is better
#df_400m['rank_total_WA_score'] = df_400m['Total_WA_Score_400m'].rank(ascending=True)  # Higher Score is better
df_400m['rank_avg_WA_score'] = df_400m['Avg_WA_Score_400m'].rank(ascending=True)  # Higher Score is better


#Final ranking
df_400m['final_rank'] = df_400m[['rank_personal_best','rank_average_sb', 'rank_career_avg', 'rank_consistency', 'rank_years_competed', 'rank_total_races','rank_races_per_year','rank_avg_place_per_race','rank_avg_WA_score']].mean(axis=1)

In [ ]:
df_400m.sort_values('final_rank').head(10)

In [ ]:
Final_Rankings_400m = df_400m.sort_values('final_rank')

In [ ]:
Final_Rankings_400m = pd.DataFrame(Final_Rankings_400m)
Final_Rankings_400m.head(3)

In [ ]:
#Let's make the entire dataframe a figure factory table


fig =  ff.create_table(Final_Rankings_400m)
fig.show()

#fig.write_html("Final_Rankings_400m.html")
#fig.write_image("Final_Rankings_400m.svg")

Scatter Plot Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Years_Competed_400m', y='Avg_Season_Best_400m', data=df_400m, s=100, hue='final_rank', palette='coolwarm')
plt.title('Years Competed vs. Career Average 100m Time')
plt.xlabel('Years Competed')
plt.ylabel('Average 400m Time (s)')
plt.show()

#plt.savefig('SB_avg_400_vs_yrs_competed_ranked.png', format='png', dpi=300)
#plt.savefig('SB_avg_400_vs_yrs_competed_ranked.jpg', format='jpg', dpi=300)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Years_Competed_400m', y='Career_Avg_400m', data=df_400m, s=100, hue='final_rank', palette='coolwarm')
plt.title('Years Competed vs. Career Average 400m Time')
plt.xlabel('Years Competed')
plt.ylabel('Career Average 400m Time (s)')
plt.show()

#plt.savefig('Career_avg_400_vs_yrs_competed_ranked.png', format='png', dpi=300)
#plt.savefig('Career_avg_400_vs_yrs_competed_ranked.jpg', format='jpg', dpi=300)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Years_Competed_400m', y='400_PR', data=df_400m, s=100, hue='final_rank', palette='coolwarm')
plt.title('Years Competed vs. 400m PR')
plt.xlabel('Years Competed')
plt.ylabel('400m PR (s)')
plt.show()

#plt.savefig('400_PR_vs_yrs_competed_ranked.png', format='png', dpi=300)
#plt.savefig('400m_PR_vs_yrs_competed_ranked.jpg', format='jpg', dpi=300)

Interactive Scatter Plot

In [ ]:
# @title Years Competed vs. Career Average 400m Time

df_400m = px.data.iris()
fig = px.scatter(df_400m, x="Years_Competed_400m", y="Avg Season Best 400m", color="final_rank",
                 size='Years_Competed_400m', hover_data=['Avg Season Best 400m']) #Potentially switch out career average for personal record regarding hover data.
fig.show()

#fig.write_html("avg_400_vs_yrs_competed_ranked.html")
#fig.write_image("avg_400_vs_yrs_competed_ranked.svg")

In [ ]:
# @title Years Competed vs. Career Average 400m Time

df_400m = px.data.iris()
fig = px.scatter(df_400m, x="Years_Competed_400m", y="Career_Avg_400m", color="final_rank",
                 size='Years_Competed_400m', hover_data=['Career_Avg_400m']) #Potentially switch out career average for personal record regarding hover data.
fig.show()

#fig.write_html("Career_avg_400_vs_yrs_competed_ranked.html")
#fig.write_image("Career_avg_400_vs_yrs_competed_ranked.svg")

In [ ]:
# @title Years Competed vs. 400m PR

df_400m = px.data.iris()
fig = px.scatter(df_400m, x="Years_Competed_400m", y="400_PR", color="final_rank",
                 size='Years_Competed_400m', hover_data=['400_PR'])
fig.show()

#fig.write_html("400m_PR_vs_yrs_competed_ranked.html")
#fig.write_image("400m_PR_vs_yrs_competed_ranked.svg")

In [ ]:
print(df_400m[['Athlete', 'Avg_Season_Best_400m', 'Years_Competed_400m', '400_PR', 'residuals', 'z_combined', 'efficiency_score','efficiency_score_sb', 'efficiency_score_pr' 'final_rank']])

In [ ]:
df_400m_stat_analysis = df_400m[['Athlete', 'Avg_Season_Best_400m', 'Years_Competed_400m', '400_PR', 'residuals', 'z_combined', 'efficiency_score','efficiency_score_sb', 'efficiency_score_pr' 'final_rank']]

In [ ]:
df_400m_stat_analysis = df_400m_stat_analysis.sort_values(by='final_rank')
df_400m_stat_analysis.head(3)

In [ ]:
#What sample size of the dataframe to we want to make into a figure factory table
df_400m_stat_analysis_sample = df_400m_stat_analysis[1:10]

#Cusomize Colors (Add colorscale=colorscale in parentheses of ff.create table)
#colorscale = [[0, '#4d004c'],[.5, '#f2e5ff'],[1, '#ffffff']]
#Cusomize Font Colors (Add font_colors=font in parentheses of ff.create table)
#font=['#FCFCFC', '#00EE00', '#008B00', '#004F00', '#660000', '#CD0000', '#FF3030']

table_data = df_400m_stat_analysis


fig =  ff.create_table(df_400m_stat_analysis_sample)
fig.show()

#fig.write_html("df_400m_stat_analysis_sample_ff.html")
#fig.write_image("df_400m_stat_analysis_sample_ff.svg")


In [ ]:
fig =  ff.create_table(df_400m_stat_analysis)
fig.show()

#fig.write_html("df_400m_stat_analysis_ff.html")
#fig.write_image("df_400m_stat_analysis_ff.svg")

In [ ]:
fig =  ff.create_table(df_400m)
fig.show()

#fig.write_html("df_400m_ff.html")
#fig.write_image("df_400m_ff.svg")

In [ ]:
df_400m = pd.DataFrame(df_400m)

In [ ]:
df_400m_stat_analysis = pd.DataFrame(df_400m_stat_analysis)

# **Let's put all the data frames created into an excel workbook**

In [ ]:
xlwriter = pd.ExcelWriter('400M_Analysis.xlsx')
df_400m.to_excel(xlwriter, sheet_name='400m')
Z_Score_Ranking_400m.to_excel(xlwriter, sheet_name='400m Z-Score Rankings')
Final_Rankings_400m.to_excel(xlwriter, sheet_name='400m Final Rankings')
#df_400m_stat_analysis.to_excel(xlwriter, sheet_name='400m Statisitcal Analysis')
#athlete_category_summary_400m.to_excel(xlwriter, sheet_name='')
#athlete_category_year_summary_400m.to_excel(xlwriter, sheet_name='')
#athlete_year_summary_400m.to_excel(xlwriter, sheet_name='')
#country_category_summary_400m.to_excel(xlwriter, sheet_name='')
#country_year_summary_400m.to_excel(xlwriter, sheet_name='')
#continent_category_summary_400m.to_excel(xlwriter, sheet_name='')
#continent_year_summary_400m.to_excel(xlwriter, sheet_name='')
#athlete_country_year_category_400m.to_excel(xlwriter, sheet_name='')
xlwriter.close()

Let's get the age of each athlete per race

In [ ]:
import pandas as pd

# Convert to datetime
df['date'] = pd.to_datetime(df['date'])
df['DOB'] = pd.to_datetime(df['DOB'])

# Age in completed years at race date
df['age_at_race'] = (
    df['date'].dt.year - df['DOB'].dt.year
    - (
        (df['date'].dt.month < df['DOB'].dt.month)
        | (
            (df['date'].dt.month == df['DOB'].dt.month)
            & (df['date'].dt.day < df['DOB'].dt.day)
        )
    )
)